# 🧱 Module 16 — Data Structures from Scratch

## 🎯 What you will learn
1. Stack (LIFO) & Queue (FIFO), deque, priority queue
2. Linked lists (singly & doubly)
3. Binary trees & Binary Search Trees (BST); traversals
4. Heaps
5. Hash tables (how dict works)
6. Graphs: adjacency list, BFS, DFS, shortest path (Dijkstra)
7. Choosing structures; Python built-in equivalents
8. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐ (stack/queue/linked list programs are common in practicals)

---
## 1. Stack — Last In, First Out (LIFO)
Think of a pile of plates. Operations: `push`, `pop`, `peek`, `is_empty`. All O(1).
Uses: undo/redo, function call stack, bracket matching, expression evaluation, browser back button, DFS.

In [1]:
class Stack:
    def __init__(self):
        self._data = []
    def push(self, x): self._data.append(x)
    def pop(self):
        if self.is_empty(): raise IndexError("pop from empty stack")
        return self._data.pop()
    def peek(self):
        if self.is_empty(): raise IndexError("peek at empty stack")
        return self._data[-1]
    def is_empty(self): return not self._data
    def __len__(self): return len(self._data)
    def __repr__(self): return f"Stack({self._data})"

s = Stack()
for x in (1, 2, 3): s.push(x)
print(s, s.pop(), s.peek(), len(s))

# Application 1: reverse a string
def reverse(text):
    st = Stack()
    for ch in text: st.push(ch)
    return "".join(st.pop() for _ in range(len(st)))
print(reverse("stack"))

# Application 2: evaluate a postfix expression  "3 4 + 2 *"  = (3+4)*2
def eval_postfix(expr):
    st = []
    for tok in expr.split():
        if tok in "+-*/":
            b, a = st.pop(), st.pop()
            st.append({"+": a + b, "-": a - b, "*": a * b, "/": a / b}[tok])
        else:
            st.append(float(tok))
    return st[0]
print(eval_postfix("3 4 + 2 *"), eval_postfix("5 1 2 + 4 * + 3 -"))

# Application 3: infix -> postfix (Shunting-yard)
def to_postfix(expr):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}
    out, ops = [], []
    for tok in expr.split():
        if tok.isdigit(): out.append(tok)
        elif tok == "(": ops.append(tok)
        elif tok == ")":
            while ops[-1] != "(": out.append(ops.pop())
            ops.pop()
        else:
            while ops and ops[-1] != "(" and prec[ops[-1]] >= prec[tok]: out.append(ops.pop())
            ops.append(tok)
    return " ".join(out + ops[::-1])
print(to_postfix("( 3 + 4 ) * 2"), "|", to_postfix("1 + 2 * 3"))

Stack([1, 2]) 3 2 2
kcats
14.0 14.0
3 4 + 2 * | 1 2 3 * +


## 2. Queue — First In, First Out (FIFO)
Like a line at a ticket counter. `enqueue` (rear), `dequeue` (front). Uses: scheduling, BFS, buffers, print jobs.
Use `collections.deque` — `list.pop(0)` is O(n)!

In [2]:
from collections import deque

class Queue:
    def __init__(self): self._d = deque()
    def enqueue(self, x): self._d.append(x)
    def dequeue(self):
        if not self._d: raise IndexError("dequeue from empty queue")
        return self._d.popleft()
    def front(self): return self._d[0]
    def __len__(self): return len(self._d)
    def __repr__(self): return f"Queue({list(self._d)})"

q = Queue()
for name in ("Asha", "Ravi", "Meena"): q.enqueue(name)
print(q, q.dequeue(), q.front(), len(q))

# Circular queue with fixed capacity (classic exam question)
class CircularQueue:
    def __init__(self, k):
        self.buf, self.k, self.head, self.size = [None] * k, k, 0, 0
    def enqueue(self, x):
        if self.size == self.k: return False
        self.buf[(self.head + self.size) % self.k] = x; self.size += 1; return True
    def dequeue(self):
        if self.size == 0: return None
        x = self.buf[self.head]; self.head = (self.head + 1) % self.k; self.size -= 1; return x
cq = CircularQueue(3)
print([cq.enqueue(i) for i in range(4)], cq.dequeue(), cq.enqueue(9), cq.buf)

Queue(['Ravi', 'Meena']) Asha Ravi 2
[True, True, True, False] 0 True [9, 1, 2]


### Deque and Priority Queue

In [3]:
d = deque([1, 2, 3])
d.appendleft(0); d.append(4); d.rotate(2)
print(d, d.popleft(), d.pop())

import heapq
class PriorityQueue:
    def __init__(self): self._h, self._n = [], 0
    def push(self, item, priority):
        heapq.heappush(self._h, (priority, self._n, item)); self._n += 1    # _n keeps FIFO among ties
    def pop(self): return heapq.heappop(self._h)[2]
    def __len__(self): return len(self._h)
pq = PriorityQueue()
pq.push("write report", 2); pq.push("fix prod bug", 1); pq.push("lunch", 3); pq.push("call boss", 1)
print([pq.pop() for _ in range(len(pq))])

deque([4, 0, 1]) 3 2
['fix prod bug', 'call boss', 'write report', 'lunch']


## 3. Linked lists
A chain of **nodes**; each node stores data and a reference to the next node. No contiguous memory.

| | Array/list | Linked list |
|---|---|---|
| Access by index | O(1) | O(n) |
| Insert/delete at front | O(n) | **O(1)** |
| Insert after a known node | O(n) | **O(1)** |
| Memory | compact | extra pointer per node |

In [4]:
class Node:
    def __init__(self, data, next=None):
        self.data, self.next = data, next

class LinkedList:
    def __init__(self):
        self.head = None
        self._size = 0

    def prepend(self, data):                       # O(1)
        self.head = Node(data, self.head); self._size += 1

    def append(self, data):                        # O(n) (no tail pointer)
        node = Node(data)
        if not self.head: self.head = node
        else:
            cur = self.head
            while cur.next: cur = cur.next
            cur.next = node
        self._size += 1

    def delete(self, value):
        prev, cur = None, self.head
        while cur:
            if cur.data == value:
                if prev: prev.next = cur.next
                else: self.head = cur.next
                self._size -= 1
                return True
            prev, cur = cur, cur.next
        return False

    def find(self, value):
        cur, i = self.head, 0
        while cur:
            if cur.data == value: return i
            cur, i = cur.next, i + 1
        return -1

    def reverse(self):                             # classic interview question
        prev, cur = None, self.head
        while cur:
            cur.next, prev, cur = prev, cur, cur.next
        self.head = prev

    def __iter__(self):
        cur = self.head
        while cur:
            yield cur.data
            cur = cur.next

    def __len__(self): return self._size
    def __repr__(self): return " -> ".join(map(str, self)) + " -> None"

ll = LinkedList()
for x in (3, 2, 1): ll.prepend(x)
ll.append(4); ll.append(5)
print(ll, len(ll))
print(ll.find(4), ll.delete(3), ll.delete(99), ll)
ll.reverse(); print(ll)
print(list(ll), sum(ll), max(ll))

# Detect a cycle: Floyd's tortoise & hare
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast: return True
    return False
a, b, c = Node(1), Node(2), Node(3)
a.next, b.next = b, c
print(has_cycle(a)); c.next = a; print(has_cycle(a))

# Find the middle node
def middle(head):
    slow = fast = head
    while fast and fast.next: slow, fast = slow.next, fast.next.next
    return slow.data
lst = LinkedList(); [lst.append(i) for i in range(1, 8)]
print(middle(lst.head))

1 -> 2 -> 3 -> 4 -> 5 -> None 5
3 True False 1 -> 2 -> 4 -> 5 -> None
5 -> 4 -> 2 -> 1 -> None
[5, 4, 2, 1] 12 5
False
True
4


### Doubly linked list
Each node has `prev` and `next`: deletion of a known node is O(1) and traversal works both ways — this is how `deque` and LRU caches are built.

In [5]:
class DNode:
    def __init__(self, data): self.data, self.prev, self.next = data, None, None

class DoublyLinkedList:
    def __init__(self): self.head = self.tail = None
    def append(self, data):
        n = DNode(data)
        if not self.tail: self.head = self.tail = n
        else: n.prev, self.tail.next, self.tail = self.tail, n, n
    def remove(self, node):                        # O(1) given the node
        if node.prev: node.prev.next = node.next
        else: self.head = node.next
        if node.next: node.next.prev = node.prev
        else: self.tail = node.prev
    def forward(self):
        cur = self.head
        while cur: yield cur.data; cur = cur.next
    def backward(self):
        cur = self.tail
        while cur: yield cur.data; cur = cur.prev

dl = DoublyLinkedList()
for i in range(1, 5): dl.append(i)
dl.remove(dl.head.next)
print(list(dl.forward()), list(dl.backward()))

[1, 3, 4] [4, 3, 1]


## 4. Trees
A tree is a hierarchy: a **root**, each node has **children**; no cycles. A **binary tree** has ≤ 2 children (left, right).

Terms: root, leaf, parent/child, sibling, depth, **height**, subtree, degree.

### Binary Search Tree (BST)
Rule: left subtree values < node < right subtree values. Search/insert/delete average O(log n), worst O(n) if unbalanced.

In [6]:
class TreeNode:
    def __init__(self, key):
        self.key, self.left, self.right = key, None, None

class BST:
    def __init__(self): self.root = None

    def insert(self, key):
        self.root = self._insert(self.root, key)
    def _insert(self, node, key):
        if node is None: return TreeNode(key)
        if key < node.key: node.left = self._insert(node.left, key)
        elif key > node.key: node.right = self._insert(node.right, key)
        return node

    def search(self, key):
        node = self.root
        while node:
            if key == node.key: return True
            node = node.left if key < node.key else node.right
        return False

    def inorder(self, node="root"):      # Left, Root, Right  -> SORTED order for a BST
        node = self.root if node == "root" else node
        if node:
            yield from self.inorder(node.left)
            yield node.key
            yield from self.inorder(node.right)

    def preorder(self, node="root"):     # Root, Left, Right
        node = self.root if node == "root" else node
        if node:
            yield node.key
            yield from self.preorder(node.left)
            yield from self.preorder(node.right)

    def postorder(self, node="root"):    # Left, Right, Root
        node = self.root if node == "root" else node
        if node:
            yield from self.postorder(node.left)
            yield from self.postorder(node.right)
            yield node.key

    def level_order(self):               # BFS with a queue
        out, q = [], deque([self.root] if self.root else [])
        while q:
            n = q.popleft(); out.append(n.key)
            q.extend(c for c in (n.left, n.right) if c)
        return out

    def height(self, node="root"):
        node = self.root if node == "root" else node
        return 0 if node is None else 1 + max(self.height(node.left), self.height(node.right))

    def min(self):
        n = self.root
        while n.left: n = n.left
        return n.key

    def delete(self, key):
        self.root = self._delete(self.root, key)
    def _delete(self, node, key):
        if node is None: return None
        if key < node.key: node.left = self._delete(node.left, key)
        elif key > node.key: node.right = self._delete(node.right, key)
        else:
            if not node.left: return node.right
            if not node.right: return node.left
            succ = node.right
            while succ.left: succ = succ.left           # in-order successor
            node.key = succ.key
            node.right = self._delete(node.right, succ.key)
        return node

tree = BST()
for k in (50, 30, 70, 20, 40, 60, 80, 35):
    tree.insert(k)
print("in-order  :", list(tree.inorder()))
print("pre-order :", list(tree.preorder()))
print("post-order:", list(tree.postorder()))
print("level     :", tree.level_order())
print(tree.search(60), tree.search(65), tree.height(), tree.min())
tree.delete(30); print("after deleting 30:", list(tree.inorder()), tree.level_order())

in-order  : [20, 30, 35, 40, 50, 60, 70, 80]
pre-order : [50, 30, 20, 40, 35, 70, 60, 80]
post-order: [20, 35, 40, 30, 60, 80, 70, 50]
level     : [50, 30, 70, 20, 40, 60, 80, 35]
True False 4 20
after deleting 30: [20, 35, 40, 50, 60, 70, 80] [50, 35, 70, 20, 40, 60, 80]


```
        50
       /  \
     30    70
    /  \   / \
  20   40 60  80
       /
      35
```

## 5. Heap
A **complete binary tree** stored in a list where each parent ≤ its children (min-heap). Gives the smallest item in O(1), push/pop in O(log n). Python: `heapq` (min-heap; negate values for a max-heap).

Index rules: parent `(i-1)//2`, children `2i+1`, `2i+2`.

In [7]:
class MinHeap:
    def __init__(self): self.a = []
    def push(self, x):
        self.a.append(x); i = len(self.a) - 1
        while i > 0 and self.a[(i - 1) // 2] > self.a[i]:           # sift up
            p = (i - 1) // 2
            self.a[p], self.a[i] = self.a[i], self.a[p]; i = p
    def pop(self):
        top, last = self.a[0], self.a.pop()
        if self.a:
            self.a[0] = last; i, n = 0, len(self.a)
            while True:                                              # sift down
                l, r, m = 2*i + 1, 2*i + 2, i
                if l < n and self.a[l] < self.a[m]: m = l
                if r < n and self.a[r] < self.a[m]: m = r
                if m == i: break
                self.a[i], self.a[m] = self.a[m], self.a[i]; i = m
        return top

h = MinHeap()
for x in (7, 3, 9, 1, 5, 2): h.push(x)
print(h.a, [h.pop() for _ in range(6)])       # heap sort!

import heapq
mx = []
for x in (7, 3, 9): heapq.heappush(mx, -x)    # max-heap trick
print(-heapq.heappop(mx))

[] [1, 2, 3, 5, 7, 9]
9


## 6. Hash tables (how `dict`/`set` work)
1. Compute `hash(key)` → integer.
2. `index = hash % table_size` → the bucket.
3. **Collisions** (two keys, same bucket) are resolved by chaining (list per bucket) or open addressing (probe for the next free slot; CPython uses this).
4. When the table gets ~2/3 full it is **resized**.

In [8]:
class HashMap:
    def __init__(self, size=8):
        self.buckets = [[] for _ in range(size)]
        self.count = 0
    def _bucket(self, key): return self.buckets[hash(key) % len(self.buckets)]
    def put(self, key, value):
        b = self._bucket(key)
        for i, (k, _) in enumerate(b):
            if k == key: b[i] = (key, value); return
        b.append((key, value)); self.count += 1
        if self.count > 0.75 * len(self.buckets): self._resize()
    def get(self, key, default=None):
        for k, v in self._bucket(key):
            if k == key: return v
        return default
    def _resize(self):
        old = [kv for b in self.buckets for kv in b]
        self.buckets = [[] for _ in range(len(self.buckets) * 2)]; self.count = 0
        for k, v in old: self.put(k, v)

hm = HashMap()
for i, w in enumerate("the quick brown fox jumps over the lazy dog".split()):
    hm.put(w, i)
print(hm.get("fox"), hm.get("the"), hm.get("cat", "?"), len(hm.buckets), hm.count)

3 6 ? 16 8


## 7. Graphs
A graph = **vertices (nodes)** + **edges**. Directed/undirected, weighted/unweighted. Common representation: **adjacency list** (dict of lists).

In [9]:
graph = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}

def bfs(g, start):                       # level by level; finds the SHORTEST path (unweighted)
    seen, order, q = {start}, [], deque([start])
    while q:
        node = q.popleft(); order.append(node)
        for nb in g[node]:
            if nb not in seen:
                seen.add(nb); q.append(nb)
    return order

def dfs(g, node, seen=None):             # go deep first
    seen = set() if seen is None else seen
    seen.add(node); order = [node]
    for nb in g[node]:
        if nb not in seen: order += dfs(g, nb, seen)
    return order

def shortest_path(g, s, t):
    prev, q = {s: None}, deque([s])
    while q:
        n = q.popleft()
        if n == t: break
        for nb in g[n]:
            if nb not in prev: prev[nb] = n; q.append(nb)
    if t not in prev: return None
    path = []
    while t: path.append(t); t = prev[t]
    return path[::-1]

def connected_components(g):
    seen, comps = set(), []
    for n in g:
        if n not in seen:
            comp = dfs(g, n); seen.update(comp); comps.append(comp)
    return comps

print("BFS:", bfs(graph, "A"))
print("DFS:", dfs(graph, "A"))
print("A->F:", shortest_path(graph, "A", "F"))
print(connected_components({**graph, "X": ["Y"], "Y": ["X"]}))

# Dijkstra: shortest path in a WEIGHTED graph using a priority queue
def dijkstra(g, src):
    dist = {n: float("inf") for n in g}; dist[src] = 0
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w; heapq.heappush(pq, (dist[v], v))
    return dist
wg = {"A": [("B", 4), ("C", 1)], "B": [("D", 1)], "C": [("B", 2), ("D", 5)], "D": []}
print(dijkstra(wg, "A"))

# Topological sort (dependency order) via Kahn's algorithm
def topo(deps):
    indeg = {n: 0 for n in deps}
    for n in deps:
        for m in deps[n]: indeg[m] += 1
    q = deque(n for n, d in indeg.items() if d == 0); out = []
    while q:
        n = q.popleft(); out.append(n)
        for m in deps[n]:
            indeg[m] -= 1
            if indeg[m] == 0: q.append(m)
    return out
print(topo({"shirt": ["tie", "belt"], "tie": ["jacket"], "belt": ["jacket"], "jacket": []}))

BFS: ['A', 'B', 'C', 'D', 'E', 'F']
DFS: ['A', 'B', 'D', 'E', 'F', 'C']
A->F: ['A', 'C', 'F']
[['A', 'B', 'D', 'E', 'F', 'C'], ['X', 'Y']]
{'A': 0, 'B': 3, 'C': 1, 'D': 4}
['shirt', 'tie', 'belt', 'jacket']


## 8. Which structure? (decision table)

| Need | Use |
|---|---|
| Ordered, index access, append | `list` |
| Fast membership / uniqueness | `set` / `dict` |
| LIFO | `list` (`append/pop`) |
| FIFO / both ends | `collections.deque` |
| Always get min/max | `heapq` |
| Sorted with fast insert/search | `bisect` on list, or a BST / `sortedcontainers` |
| Key → value | `dict` |
| Count things | `Counter` |
| Fixed-type numbers, memory-lean | `array`, NumPy |
| Relationships / networks | dict-of-lists graph |
| LRU cache | `OrderedDict` / `functools.lru_cache` |

---
# 🕵️ Hidden Facts & Pro Tricks

In [10]:
# 1. Python's list IS a dynamic array (not a linked list). deque is a doubly-linked list of blocks.
# 2. LRU cache in 10 lines with OrderedDict
from collections import OrderedDict
class LRU:
    def __init__(self, cap): self.cap, self.d = cap, OrderedDict()
    def get(self, k):
        if k not in self.d: return -1
        self.d.move_to_end(k); return self.d[k]
    def put(self, k, v):
        self.d[k] = v; self.d.move_to_end(k)
        if len(self.d) > self.cap: self.d.popitem(last=False)
c = LRU(2); c.put(1, "a"); c.put(2, "b"); c.get(1); c.put(3, "c")
print(list(c.d))

# 3. Trie (prefix tree) with nested dicts - autocomplete
def trie_insert(root, word):
    for ch in word: root = root.setdefault(ch, {})
    root["$"] = True
def trie_words(node, prefix=""):
    for ch, sub in node.items():
        if ch == "$": yield prefix
        else: yield from trie_words(sub, prefix + ch)
def autocomplete(root, prefix):
    for ch in prefix:
        if ch not in root: return []
        root = root[ch]
    return sorted(trie_words(root, prefix))
T = {}
for w in ("car", "card", "care", "cat", "dog"): trie_insert(T, w)
print(autocomplete(T, "car"))

# 4. Union-Find (disjoint sets) - detect connected groups fast
parent = list(range(6))
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b): parent[find(a)] = find(b)
union(0, 1); union(1, 2); union(3, 4)
print(find(0) == find(2), find(0) == find(3))

# 5. Nested list as a tree: use recursion or an explicit stack to traverse
# 6. Tuples in heaps compare element by element: (priority, counter, item) avoids comparing items
# 7. collections.deque(maxlen=n) is a free sliding window / "last n items" buffer
recent = deque(maxlen=3)
for i in range(6): recent.append(i)
print(list(recent))

# 8. sortedcontainers (pip) offers SortedList/SortedDict with O(log n) operations
# 9. array module and memoryview: compact numeric storage
import array
a = array.array("i", [1, 2, 3]); print(a.itemsize, a.tobytes())
# 10. graphlib.TopologicalSorter is a built-in for the dependency problem above
# 11. A queue of tasks can be shared between threads safely: queue.Queue (blocking, thread-safe)
import queue
tq = queue.Queue(); tq.put("job"); print(tq.get())
# 12. Bloom filters, skip lists, B-trees: what databases use — worth reading about later

[1, 3]
['car', 'card', 'care']
True False
[3, 4, 5]
4 b'\x01\x00\x00\x00\x02\x00\x00\x00\x03\x00\x00\x00'
job


**Hidden fact:** Python's `dict` uses *open addressing with perturbed probing* and separate index/entries arrays ("compact dict") which is why it preserves order and uses less memory than the older design.

**Hidden fact:** a plain `list` can act as a heap, stack, or queue; most "advanced" structures in interviews are compositions of these plus a dict.

---
# 📝 Exam Corner

**Theory**
1. Define stack and queue; list their operations and applications.
2. Compare array (list) and linked list.
3. What is a BST? Explain in-order, pre-order, post-order traversals for the tree above.
4. Difference between BFS and DFS (queue vs stack, shortest path).
5. How does a hash table resolve collisions?
6. What is a priority queue / heap?

**Trace:** for the BST above, the traversals are
- In-order: 20 30 35 40 50 60 70 80
- Pre-order: 50 30 20 40 35 70 60 80
- Post-order: 20 35 40 30 60 80 70 50

**Output prediction**

In [11]:
st = []
st.append(1); st.append(2); st.append(3)
st.pop()
st.append(4)
print(st, st[-1])

q = deque([1, 2, 3])
q.rotate(1)
q.append(q.popleft())
print(list(q))

print(list(BST().inorder()))

[1, 2, 4] 4
[1, 2, 3]
[]


<details><summary>💡 Answers</summary>

```
[1, 2, 4] 4
[1, 2, 3]   (rotate(1) -> [3,1,2]; move the front to the back -> [1,2,3])
[]
```
</details>

---
## ✅ Summary
- Stack = LIFO (`list`), Queue = FIFO (`deque`), Priority queue = heap (`heapq`).
- Linked lists: O(1) inserts at known positions, O(n) access. BST: sorted, O(log n) if balanced.
- Hash tables give O(1) average lookups (dict/set). Graphs: adjacency lists + BFS/DFS/Dijkstra.

➡️ **Next:** Module 17 — Regular Expressions & Text Processing.